# In-class Session: Materials Project · Experimental Databases · Phase Diagrams
AI for Materials Science — Hands-on session 1

Today we will search a computational database for materials that meet a set of conditions, pull in
some experimental data, and finally build a phase diagram from calculated energies.
The figure and table you produce along the way are what you submit for attendance.

## Today's plan

### 0 · Setup
Install the libraries, download the course data, and enter your MP API key.

### A · Querying the Materials Project
Learn to turn a question into search conditions, and collect the matching materials into a table.

### B · Experimental databases through matminer
Load experimental datasets from different sources through one interface, and check row counts and
missing values.

### C · Phase diagrams
Build the Li–Fe–O convex hull from calculated energies and read the resulting figure.

### D · Attendance submission
Save your table and figure under filenames tagged with your student ID.

---

Run the cells one at a time from the top.
Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not
defined" error.
Lines starting with `##` inside the code are comments written for you; Python does not run them.

MP API examples that we do not cover here — structures, electronic structure, batteries, surfaces,
aqueous stability and more — are collected in
`2026_2_Hands_on_session1_api_examples.ipynb` in the same folder.

## 0. Setup

Four things to get in place: the libraries, the course data, an output folder, and your MP API key.

### 0-1. Install the libraries
`pymatgen` handles compositions, structures and phase diagrams, `mp_api` queries the Materials
Project, and `matminer` loads experimental datasets.
NumPy, pandas and matplotlib come along with them.

In [ ]:
## A leading ! runs a terminal command instead of Python.
!pip install -q pymatgen mp_api matminer

### 0-2. Download the course material
All of today's data lives in the `Data/` folder of the course repository.
The calculated entries (`Li-Fe-P-O_entries.json`) and the four experimental datasets come down
together, so there are no separate downloads during class.

If you run this a second time you will see a "destination path already exists" message. That is
harmless.

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries
Installing and importing are two different steps. Installing puts files on the machine; `import`
brings a tool into this notebook.

In [ ]:
## Standard Python tools for paths, timestamps, key entry, and inspecting function signatures.
import json
import os
import inspect
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

## np for numeric work, pd for tables, plt for figures.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Restoring saved calculation data, and the pymatgen tools for compositions and phase diagrams.
from monty.serialization import loadfn
from pymatgen.core import Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter

## The MP query client, and the matminer helpers that load experimental datasets.
from mp_api.client import MPRester
from mp_api.client.routes.materials.summary import SummaryRester
from matminer.datasets import load_dataset, get_available_datasets, get_dataset_description

### 0-4. Set the file paths
We decide once which folder to read from and where results go.
Running the cell prints the list of files inside `Data/`.
If you get an error instead of a list, check that you ran the `git clone` cell in 0-2.

In [ ]:
## The Data folder inside the repository downloaded in 0-2.
DATA_DIR = Path("MS49900-AI4M/Data")

## Every table, figure and submission file from today is saved here.
OUTPUT = Path("outputs/02_inclass")
OUTPUT.mkdir(parents=True, exist_ok=True)

sorted(path.name for path in DATA_DIR.iterdir())

### 0-5. Enter your MP API key
Section A and part of section C query the Materials Project directly.
You can get a key for free from the [MP account page](https://next-gen.materialsproject.org/api).

Treat the key like a password. Written into code or a submitted file, it is exposed.
`getpass` takes the input without echoing it to the screen.

In [ ]:
## Use the key from an environment variable if it is set; otherwise ask for it directly.
API_KEY = os.getenv("MP_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("Materials Project API key: ").strip()

if not API_KEY:
    raise ValueError("An MP API key is required. Get one, then rerun this cell.")

## Confirm the connection works and note which database version we are querying.
with MPRester(API_KEY) as mpr:
    print("MP DB version:", mpr.db_version)

## A. Finding materials that meet your conditions in the Materials Project

Using a materials database really comes down to **turning your question into search conditions**.
Even for something as simple as "show me LiFePO₄", the condition you need differs depending on
whether you want the structures at that composition or every compound containing those elements.

### First, three terms worth separating

- **material** — one representative crystal structure. It carries an ID such as `mp-19017`.
- **task** — one individual DFT calculation. A single material has several tasks behind it.
- **entry** — an object holding a composition together with a calculated energy. This is what phase
  diagrams are built from.

The same composition can correspond to several materials when the atoms are arranged differently.
And you should not assume that the band gap and the energy shown in a summary came from the **same
task**.

### A-1. Conditions change with the question
Work out what each of these five covers. We will use four of them directly in this section.

- `formula="LiFePO4"` — every structure at that composition. Several rows if there are polymorphs.
- `chemsys="Li-Fe-P-O"` — compounds made of exactly these four elements.
- `elements=["Li","Fe","P","O"]` — anything containing these four. Extra elements are allowed.
- `get_entries_in_chemsys([...])` — **all subsystems**, down to elements and binaries. Required for
  phase diagrams.
- `formula="AB"` — a ratio without naming the elements (an anonymous formula). Useful for sweeping
  1:1 compounds.

Full syntax is in
[Querying data](https://docs.materialsproject.org/downloading-data/using-the-api/querying-data) and
[API examples](https://docs.materialsproject.org/downloading-data/using-the-api/examples).

### A-2. Seeing the scope of three conditions
Before asking the server, let us count how many entries each condition selects using the calculation
data we already have.
The file here holds 859 calculated Li–Fe–P–O entries published with pymatgen.

Notice how far apart the numbers are. Pick the wrong condition and you are looking at a completely
different set from the one you meant.

In [ ]:
## loadfn restores a saved JSON file back into the original list of entry objects.
reference_entries = loadfn(DATA_DIR / "Li-Fe-P-O_entries.json")

print("Entries in the calculation data:", len(reference_entries))
print("First entry:", reference_entries[0].entry_id, reference_entries[0].composition.reduced_formula)

In [ ]:
## Collect the element symbols of one entry into a set, with duplicates removed.
def element_set(entry):
    return {element.symbol for element in entry.composition.elements}

lfp_set = {"Li", "Fe", "P", "O"}

## == tests for exactly the same element set; <= tests for a subset.
formula_hits = [e for e in reference_entries if e.composition.reduced_formula == "LiFePO4"]
exact_hits = [e for e in reference_entries if element_set(e) == lfp_set]
subsystem_hits = [e for e in reference_entries if element_set(e) <= lfp_set]

pd.DataFrame([
    {"condition": "formula LiFePO4", "entries selected here": len(formula_hits)},
    {"condition": "exactly Li-Fe-P-O", "entries selected here": len(exact_hits)},
    {"condition": "all subsystems too", "entries selected here": len(subsystem_hits)},
])

### A-3. Checking what you can ask for and what you can get back
Before writing a query, it pays to look at **the conditions it accepts** and **the fields it can
return**.

`inspect.signature` lists the argument names a function takes. No server connection needed.
`available_fields` is the list of fields you can actually retrieve, so that one does ask the server.

In [ ]:
## List the condition names the query function accepts.
print("Conditions accepted by summary.search:")
print(list(inspect.signature(SummaryRester.search).parameters))

In [ ]:
## Leaving the with block closes the connection.
with MPRester(API_KEY) as mpr:
    fields = mpr.materials.summary.available_fields

print("Number of fields available from summary:", len(fields))
print(fields[:20], "...")

### A-4. A helper that turns search results into a table
MP returns results as a list of document objects. Unpacking those by hand every time is tedious, so
we write one helper that pulls out the fields we need as a DataFrame and reuse it throughout section
A.

Units are written into the column names: band gap in eV, energies in eV/atom, density in g/cm³.

In [ ]:
## The fields to request. Without this, far more comes back than we need.
SUMMARY_FIELDS = ["material_id", "formula_pretty", "chemsys", "nelements", "nsites",
                  "band_gap", "energy_above_hull", "formation_energy_per_atom",
                  "is_stable", "density", "symmetry"]

## Column names for the table. Putting units in the name saves confusion later.
FRAME_COLUMNS = ["material_id", "formula", "chemsys", "nelements", "nsites", "band_gap_eV",
                 "e_hull_eV_atom", "formation_eV_atom", "is_stable", "density_g_cm3",
                 "spacegroup", "spacegroup_number"]

In [ ]:
## Take a list of documents and return a table with one material per row.
def summary_to_frame(docs):
    rows = []
    for doc in docs:
        ## symmetry is not a single value but an object holding a symbol and a number, so we go one level deeper.
        sym = doc.symmetry
        rows.append(dict(zip(FRAME_COLUMNS, [
            str(doc.material_id), doc.formula_pretty, doc.chemsys, doc.nelements, doc.nsites,
            doc.band_gap, doc.energy_above_hull, doc.formation_energy_per_atom,
            doc.is_stable, doc.density,
            getattr(sym, "symbol", None), getattr(sym, "number", None),
        ])))
    return pd.DataFrame(rows, columns=FRAME_COLUMNS)

### A-5. Querying the structures at the LiFePO₄ composition
Our first query. We ask for every structure at this composition with `formula="LiFePO4"` and sort by
distance to the hull.

`num_chunks=1, chunk_size=200` means "return at most 200 records in one go".
So **do not read the row count as the total number of matches in the database**.
If you get exactly 200 rows, suspect there are more.

In [ ]:
with MPRester(API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        formula="LiFePO4", fields=SUMMARY_FIELDS,
        all_fields=False, num_chunks=1, chunk_size=200)

lfp_df = summary_to_frame(docs)
lfp_df.to_csv(OUTPUT / "mp_lfp_summary.csv", index=False)

print("Rows returned:", len(lfp_df))
lfp_df.sort_values("e_hull_eV_atom").head(10)

### A-6. Stacking conditions to narrow candidates: 1:1 binary oxides
This time we stack several conditions the way a real screening would.
We are after **1:1 binary oxides within 0.05 eV/atom of the hull with a band gap between 1 and 4 eV**.

- `chemsys="*-O"` — binary systems containing oxygen
- `formula="AB"` — the two elements in a 1:1 ratio
- `energy_above_hull=(0, 0.05)` — on or very near the hull
- `band_gap=(1, 4)` — the semiconducting range

A small `energy_above_hull` only means the material is stable against decomposition *within this
calculation model*.
Whether it can actually be synthesised is a separate question.

In [ ]:
with MPRester(API_KEY) as mpr:
    ao_docs = mpr.materials.summary.search(
        chemsys="*-O", formula="AB", num_elements=2,
        energy_above_hull=(0, 0.05), band_gap=(1, 4),
        fields=SUMMARY_FIELDS, all_fields=False, num_chunks=1, chunk_size=200)

ao_df = summary_to_frame(ao_docs)
ao_df.to_csv(OUTPUT / "mp_AO_candidates.csv", index=False)

print("Candidates passing the conditions:", len(ao_df))
ao_df.sort_values(["e_hull_eV_atom", "band_gap_eV"]).head(15)

### A-7. When server-side conditions are not enough: R-3m LiXO₂
`formula="ABC2"` only says "three elements in a 1:1:2 ratio". It **cannot say which element is the 1
and which is the 2.**
A composition with 2 Li and 1 O matches just as well.

So we do it in two stages: cast a wide net on the server, then filter what comes back using pymatgen
`Composition`.
It is a good illustration of where the API stops and your own code takes over.

In [ ]:
## Reduce the formula to its simplest ratio, then check for exactly 1 Li and 2 O.
def is_lixo2(formula):
    comp = Composition(formula).reduced_composition
    ## np.isclose allows for floating-point error when testing "equal".
    return (len(comp.elements) == 3 and np.isclose(comp["Li"], 1)
            and np.isclose(comp["O"], 2) and np.isclose(comp.num_atoms, 4))

## See how the function judges a few examples before using it on real results.
{f: is_lixo2(f) for f in ["LiCoO2", "LiFeO2", "Li2FeO", "LiFePO4"]}

In [ ]:
with MPRester(API_KEY) as mpr:
    ## Narrow on the server first: stable ternary ABC2 candidates in space group 166 (R-3m).
    lxo_docs = mpr.materials.summary.search(
        elements=["Li", "O"], num_elements=3, formula="ABC2",
        spacegroup_number=166, is_stable=True, fields=SUMMARY_FIELDS,
        all_fields=False, num_chunks=1, chunk_size=200)

lxo_df = summary_to_frame(lxo_docs)
## map applies the test to every value in the formula column; loc keeps the rows where it is True.
lixo2_df = lxo_df.loc[lxo_df["formula"].map(is_lixo2)].copy()
lixo2_df.to_csv(OUTPUT / "mp_stable_R3m_LiXO2.csv", index=False)

print(f"Of the {len(lxo_df)} ABC2 candidates the server returned, {len(lixo2_df)} are really LiXO2.")
lixo2_df

### A-8. Try it yourself
Rerun the AO search in A-6 with the lower band gap bound changed from `1.0` to `2.0`.

- How many candidates do you get now?
- What are the top three material IDs?
- What kind of materials dropped out when you tightened the condition?

If the queries are not working, move on to section B first and sort out section A with a TA.

## B. Loading experimental data with matminer

Everything in section A was **calculated**. Now we turn to **measured** values collected from the
literature.

matminer gives you one way (`load_dataset`) to reach experimental datasets that are scattered across
different sources.
matminer is not itself a measurement database; think of it as a single counter serving data of very
mixed origins.

Four datasets today.

- `steel_strength` — 312 steels. Composition plus yield strength, tensile strength and elongation.
  Strengths in MPa, composition in wt%
- `ucsb_thermoelectrics` — 1,093 thermoelectric records. **The same composition measured at a
  different temperature is a different row**
- `matbench_expt_gap` — 4,604 experimental band gaps. Distributed by MP, but measured, not DFT
- `citrine_thermal_conductivity` — 872 thermal conductivity records. Units and measurement
  conditions are mixed in as free text

The point of this section is less the property values themselves and more the habit of asking
**"can I take this table at face value and start computing?"**

### B-1. Seeing what is available
`get_available_datasets` lists the names and `get_dataset_description` explains one of them.
Checking whether data on your topic already exists is where any project starts.

In [ ]:
available = get_available_datasets(print_format=None)
print("Datasets available through matminer:", len(available))

## any is True if at least one of the three search words appears in the name.
print([name for name in available
       if any(word in name for word in ["expt", "steel", "thermoelectric"])])

In [ ]:
print(get_dataset_description("ucsb_thermoelectrics"))

### B-2. Loading all four at once
Give `load_dataset` a name and you get a table. The source does not change how you call it.

We point `data_home` at the folder downloaded in 0-2 and pass `download_if_missing=False`, meaning
"do not go back to the internet during class".

In [ ]:
DATASET_NAMES = ["steel_strength", "ucsb_thermoelectrics",
                 "matbench_expt_gap", "citrine_thermal_conductivity"]

## Collecting the tables into a dictionary lets us pull each one out by name later.
experimental = {}
for name in DATASET_NAMES:
    experimental[name] = load_dataset(name, data_home=str(DATA_DIR), download_if_missing=False)

## shape is (number of rows, number of columns).
{name: frame.shape for name, frame in experimental.items()}

### B-3. Summarising all four in one table
Before looking at any dataset in detail, let us compare size, number of compositions, missing values
and duplicates side by side.
This is a good first table to build whenever you receive new data.

In [ ]:
inventory = []
for name, frame in experimental.items():
    ## The composition column is named either composition or formula depending on the dataset.
    formula_col = "composition" if "composition" in frame else "formula"
    inventory.append({
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        ## nunique counts distinct values, ignoring repeats.
        "unique_composition_strings": frame[formula_col].nunique(),
        ## Two sums over isna() give the number of blank cells in the whole table.
        "missing_cells": int(frame.isna().sum().sum()),
        ## duplicated() counts rows that are identical across every column.
        "fully_duplicated_rows": int(frame.duplicated().sum()),
    })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv(OUTPUT / "experimental_dataset_inventory.csv", index=False)
inventory_df

### B-4. Row count and number of valid measurements are different things
`len(df)` counts all rows. `df.count()`, on the other hand, counts **per column, excluding blanks**.
Seeing where those two diverge is the heart of this section.

Let us open the steel data and look at its columns first.

In [ ]:
steel = experimental["steel_strength"]
steel.head()

In [ ]:
## info shows the dtype of each column together with how many rows hold a value.
steel.info()

In [ ]:
## Put valid and missing counts side by side, column by column.
pd.DataFrame({"valid_measurements": steel.count(), "missing": steel.isna().sum()})

`describe()` gives count, mean, standard deviation and quartiles in one go.
Read the `count` row first here too. A smaller count for elongation means that column has blanks.

In [ ]:
## Double brackets select just the three numeric columns to summarise.
steel[["yield strength", "tensile strength", "elongation"]].describe()

### B-5. Why the same composition appears many times
In the thermoelectric data, the same composition **measured at a different temperature is a separate
record**.
So deciding "same composition, must be a duplicate" and dropping rows would corrupt the dataset.

Below, count rows that merely share a composition separately from rows that are identical throughout,
and compare.

In [ ]:
te = experimental["ucsb_thermoelectrics"]
te.head()

In [ ]:
## With subset, only those columns are compared; without it, every column must match.
print("Rows repeating a composition:", int(te.duplicated(subset=["composition"]).sum()))
print("Rows identical in every column:", int(te.duplicated().sum()))

## dropna removes a row if any of the listed columns is blank.
print("Total rows:", len(te))
print("Rows with both temperature and zT:", len(te.dropna(subset=["T [K]", "zT"])))

### B-6. Stored dtype and statistical meaning are not the same
A pandas dtype tells you **how a value is stored on the machine**. It does not tell you **how the
value should be treated statistically**.

- **Nominal** — a name or code with no order (a synthesis method)
- **Ordinal** — ordered, but with no guarantee of even spacing (Mohs hardness)
- **Discrete** — a countable integer quantity (number of atoms in a structure)
- **Continuous** — any real value within a range (temperature, a measured property)

Run the two cells below, then discuss with a partner where each of `crystallinity`, `synthesis`,
`spacegroup`, `T [K]` and `zT` belongs.

In particular, `spacegroup` is stored as a number. **Would its mean mean anything to a materials
scientist?**

In [ ]:
## dtypes lists the stored type of each column.
te.dtypes.rename("pandas dtype").to_frame()

In [ ]:
## Look at the stored type together with the actual values. T [K] is absolute temperature; zT is dimensionless.
te[["crystallinity", "synthesis", "spacegroup", "T [K]", "zT"]].head()

### B-7. Fix the measurement conditions before comparing
You cannot simply take "the material with the highest zT". If the temperatures differ, it is not a
fair comparison.

So we first **restrict to 300–500 K** and then take the maximum zT per composition within that window.
That value is "the highest observed in this window", not "a comparison of all materials at the same
temperature".

In [ ]:
## Drop rows missing a required column, then use between to select 300-500 K inclusive.
te_valid = te.dropna(subset=["T [K]", "zT"]).copy()
te_window = te_valid.loc[te_valid["T [K]"].between(300, 500)]

## groupby collects rows sharing a composition; agg says what to compute for each group.
top_te = (te_window.groupby("composition", as_index=False)
          .agg(n_measurements=("zT", "size"), max_observed_zT=("zT", "max"),
               lowest_T=("T [K]", "min"), highest_T=("T [K]", "max"))
          .sort_values("max_observed_zT", ascending=False))

print(f"{len(te_window)} measurements between 300-500 K, across {len(top_te)} compositions")
top_te.head(10)

In [ ]:
## fig is the whole figure, ax is the plotting area. We scatter every record, before the temperature cut.
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.scatter(te_valid["T [K]"], te_valid["zT"], s=12, alpha=0.35)
ax.set(xlabel="Measurement temperature (K)", ylabel="Experimental zT",
       title="UCSB thermoelectrics: measured records")

fig.tight_layout()
fig.savefig(OUTPUT / "ucsb_zT_vs_T.png", dpi=160)
plt.show()

### B-8. A look at the experimental band gap distribution
`matbench_expt_gap` holds band gaps obtained by measurement.
MP distributes it, but these are **experimental values, not DFT results**. Do not mix them with the
calculated band gaps from section A.

Look at the minimum in the distribution. A lot of zeros means entries measured as metallic are
included.

In [ ]:
gap = experimental["matbench_expt_gap"]

## rename puts the unit into the name of the summary.
gap["gap expt"].describe().rename("Experimental gap (eV)")

### B-9. Decide what counts as a duplicate first
The thermal conductivity data carries units and measurement conditions **as text**.
Columns like these need checking before any numeric column goes into a calculation.

In [ ]:
thermal = experimental["citrine_thermal_conductivity"]

## value_counts counts rows per value. dropna=False counts blanks too.
thermal["k-units"].value_counts(dropna=False)

In [ ]:
## Select the two condition columns together and keep each combination once, to see what was recorded.
thermal[["k_condition", "k_condition_units"]].drop_duplicates().head(12)

In [ ]:
## Count fully identical rows separately from rows that merely share a formula.
full_duplicate_count = int(thermal.duplicated().sum())
formula_repeat_count = int(thermal.duplicated(subset=["formula"]).sum())

print("Rows identical in every column:", full_duplicate_count)
print("Rows merely sharing a formula:", formula_repeat_count)
print("Matches the reference value of 18 for this release:", full_duplicate_count == 18)

Rows that are identical throughout can be cleaned up once you have checked their provenance.
Rows that only share a `formula`, however, may be separate records taken at different temperatures or
conditions, so they must not be dropped wholesale.

### B-10. Try it yourself
Extract the UCSB records with **500–700 K and zT ≥ 0.8** and save them as a CSV.

Four numbers to report:

1. the total number of rows
2. the number of rows after dropping missing values in the required columns
3. the number of rows passing both conditions
4. the number of distinct composition strings among those rows

Then write one sentence on **why you kept multiple rows for the same composition**.

The cell below sets up the pieces. This is not a fill-in-the-blank exercise; carry on writing it
yourself.
If you get stuck, look back at the `dropna(subset=...)` and boolean masks used in B-5 and B-7.

In [ ]:
## Work on a copy so the original te is left intact.
task_input = te.copy()
required_cols = ["composition", "T [K]", "zT"]
task_temperature_range = (500, 700)
task_min_zT = 0.8
task_output_csv = OUTPUT / "task_ucsb_500_700K.csv"

## Order: len() -> dropna(subset=...) -> two condition masks -> nunique() -> to_csv()
## Write your solution from here.
task_input[required_cols].head()

## C. Building a phase diagram from calculated energies

This is the core of the session. We work through how calculated energies per composition tell you
**which phases are stable**.

We are building the Li–Fe–O ternary phase diagram. Change the elements and the same code gives you
Li–Co–O.

### Why querying `chemsys="Li-Fe-O"` is not enough
Fetching only the strictly ternary compounds leaves out the **elemental references** such as Li, Fe
and O₂, and the **competing binaries** such as Li₂O and Fe₂O₃.
Without anything to compare against you cannot build a convex hull. Restricting to `is_stable=True`
has the same problem: with no unstable phases, you cannot measure *how* unstable something is.

So a phase diagram needs **every subsystem — elements, binaries and ternaries alike**.
That is the job of `get_entries_in_chemsys`, which we met in A-1.

### C-1. Selecting the entries for the phase diagram
Today we take the Li·Fe·O subsystem out of the calculation data we already read in A-2.
Because the dataset is version-pinned, everyone in the room sees the same figure.

Check that **all three elemental references are present** in the output. If even one is missing, the
hull cannot be built.

In [ ]:
PD_ELEMENTS = ["Li", "Fe", "O"]

## Reuse the element_set helper from A-2. Keep entries whose elements fall inside Li-Fe-O.
pd_entries = [e for e in reference_entries if element_set(e) <= set(PD_ELEMENTS)]
phase_source = "pymatgen historical fixture 0428f232a569 | stored GGA/GGA+U corrections"

print("Entries in this system:", len(pd_entries))
print("Elemental references:", sorted({e.composition.reduced_formula
                                       for e in pd_entries if e.composition.is_element}))

### C-2. Computing the hull and telling three energies apart
Hand `PhaseDiagram` a list of entries and it computes the convex hull and the set of stable phases.

The table has three energy columns. **Make sure you can tell them apart.**

- `corrected_energy_eV_atom` — the per-atom energy of that calculation. Not comparable across
  materials
- `formation_energy_eV_atom` — formation energy relative to the elemental references. More negative
  means more stable than the elements
- `energy_above_hull_eV_atom` — distance to the hull. **Zero means stable in this system**; larger
  means easier to decompose

The formation energy here comes from a 0 K electronic structure calculation.
Treating the solid `pV` term as small, it works as an approximation to the 0 K formation enthalpy,
but it is not the finite-temperature Gibbs free energy.
Temperature-dependent free energies come up in notebook 03.

In [ ]:
## The convex hull is computed from the compositions and corrected energies of the input entries.
phase_diagram = PhaseDiagram(pd_entries)

phase_rows = []
for entry in pd_entries:
    phase_rows.append({
        "entry_id": str(entry.entry_id),
        "formula": entry.composition.reduced_formula,
        "n_atoms_in_entry": entry.composition.num_atoms,
        "corrected_energy_eV_atom": entry.energy_per_atom,
        "formation_energy_eV_atom": phase_diagram.get_form_energy_per_atom(entry),
        "energy_above_hull_eV_atom": phase_diagram.get_e_above_hull(entry),
        ## in tests whether this entry belongs to the set of stable phases.
        "stable_in_this_PD": entry in phase_diagram.stable_entries,
        "run_type": entry.parameters.get("run_type"),
    })

phase_df = pd.DataFrame(phase_rows).sort_values(["energy_above_hull_eV_atom", "formula"])

print("Stable entries:", int(phase_df["stable_in_this_PD"].sum()), "/", len(phase_df))
phase_df.head(20)

### C-3. Drawing the phase diagram
`PDPlotter` draws the hull on a triangle. `show_unstable=False` keeps only the stable phases.

How to read it:

- **Vertices** — pure elements
- **Edges** — binary compounds
- **Interior** — ternary compounds
- **Phases joined by a line** — combinations that can coexist at compositions in between

The figure is based on 0 K energies from a fixed calculation model. Temperature, pressure and
reaction kinetics are not in it.

In [ ]:
plotter = PDPlotter(phase_diagram, backend="matplotlib", show_unstable=False,
                    linewidth=1.5, markersize=7)
## Turn off the default labels; we place them by hand below so they do not overlap.
ax = plotter.get_plot(label_stable=False, label_unstable=False)
ax.set_aspect("equal", adjustable="box")

phase_fig = ax.figure
phase_fig.set_size_inches(8, 7)

In [ ]:
## Offsets tuned for this Li-Fe-O example so that labels on nearby phases do not collide.
## Only the label positions move; hull coordinates and computed values are untouched.
label_offsets = {"Li": (-12, -13), "Fe": (14, -13), "O2": (0, 20),
                 "Li2O": (-35, -8), "Li2O2": (-45, 10), "Li2FeO3": (-30, 27),
                 "LiFeO2": (0, 30), "Li2FeO2": (8, -25), "Li5FeO4": (-55, 12),
                 "FeO": (28, -8), "Fe3O4": (40, 5), "Fe2O3": (32, 25)}

## pd_plot_data[1] pairs each stable phase with its coordinates.
for xy, entry in plotter.pd_plot_data[1].items():
    formula = entry.composition.reduced_formula
    ## xy is the position of the phase; xytext is an on-screen offset in points.
    ax.annotate(formula, xy=xy, xytext=label_offsets.get(formula, (10, 10)),
                textcoords="offset points", ha="center", va="center", fontsize=10,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1),
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.6), annotation_clip=False)

ax.set_title("Li-Fe-O | corrected GGA/GGA+U | historical reference")
## bbox_inches="tight" keeps labels that stick out beyond the axes in the saved image.
phase_fig.savefig(OUTPUT / "phase_diagram.png", dpi=180, bbox_inches="tight")
plt.show()

### C-4. Same formula, different structures: LiFeO₂ polymorphs
The figure shows only stable phases, but the dataset actually contains several structures at the
LiFeO₂ composition.

Comparing their distances to the hull reveals **which structure is the stable one**.
It also shows what the unstable ones decompose into.

In [ ]:
same_formula = [e for e in pd_entries if e.composition.reduced_formula == "LiFeO2"]

lifeo2_rows, lifeo2_product_ids = [], set()
for entry in same_formula:
    ## This function returns the decomposition products and the hull distance together.
    decomposition, e_hull = phase_diagram.get_decomp_and_e_above_hull(entry)
    products = [f"{e.entry_id}:{e.composition.reduced_formula}" for e in decomposition]
    lifeo2_product_ids.update(str(e.entry_id) for e in decomposition)
    lifeo2_rows.append({"entry_id": str(entry.entry_id),
                        "energy_above_convex_hull_eV_atom": float(e_hull),
                        "decomposition_products": " + ".join(products)})

lifeo2_polymorphs = (pd.DataFrame(lifeo2_rows)
                     .sort_values("energy_above_convex_hull_eV_atom")
                     .reset_index(drop=True))

print("LiFeO2 polymorphs:", len(lifeo2_polymorphs))
print("Decomposition product IDs:", lifeo2_product_ids)
lifeo2_polymorphs

Two things to take from that table.

First, the hull distance in the top row is 0. That is the stable phase, the LiFeO₂ whose label appears
in the figure.
The bottom row sits about 0.34 eV/atom away. Same chemical formula, that much difference.

Second, every unstable polymorph decomposes into **the same single phase (`mp-851027`)**.
Since the composition is identical, so is what it decomposes into. The next cell shows what happens
when the composition differs.

### C-5. A different composition gives several products: LiFe₂O₄
`mp-25516` (LiFe₂O₄) lies a little above the hull and **decomposes into three phases**.

The weights returned by `get_decomp_and_e_above_hull` are **atomic fractions**, which are not the
same as reaction coefficients.
Below we convert them into coefficients per formula unit.

In [ ]:
## Take the first entry in the list matching this condition.
target = next(e for e in pd_entries if str(e.entry_id) == "mp-25516")
decomposition, target_e_hull = phase_diagram.get_decomp_and_e_above_hull(target)

## Atom count of the reduced formula. LiFe2O4 has 7.
target_atoms = Composition(target.composition.reduced_formula).num_atoms

decomposition_rows = []
for product, weight in decomposition.items():
    product_atoms = Composition(product.composition.reduced_formula).num_atoms
    decomposition_rows.append({
        "phase_entry_id": str(product.entry_id),
        "phase_formula": product.composition.reduced_formula,
        "atomic_fraction_weight": float(weight),
        ## atomic fraction x target atom count / product atom count = coefficient per formula unit
        "coefficient_per_target_formula": float(weight) * target_atoms / product_atoms,
    })

print(f"{target.entry_id} ({target.composition.reduced_formula}) "
      f"| {float(target_e_hull):.6f} eV/atom above the hull | {target_atoms:.0f} atoms")
pd.DataFrame(decomposition_rows).sort_values("phase_entry_id")

Reading off the coefficient column gives a balanced decomposition reaction.

$$\mathrm{LiFe_2O_4 \rightarrow LiFeO_2 + \tfrac{1}{2}\,Fe_2O_3 + \tfrac{1}{4}\,O_2}$$

Multiplying the atomic fractions `4/7`, `5/14` and `1/14` by `7 / product atom count` (4, 5 and 2)
gives 1, 1/2 and 1/4.

Checking the balance with a reaction object and converting to eV/reaction is covered in notebook 03.

## D. Attendance submission

Save the table and figure you built into files tagged with your student ID. Three files to submit.

1. `attendance_<id>_phase_diagram.png` — the phase diagram from C-3
2. `attendance_<id>_phases.csv` — the phase table from C-2
3. `attendance_<id>_reflection.json` — the notes you write below

### What to write
- Pick three stable phases and list them in `selected_phase_ids`.
- Pick one LiFeO₂ polymorph and record its entry ID and hull distance (see the C-4 table).
- In your own words, explain **why subsystems have to be included** in a phase diagram.
- Name one **phenomenon a 0 K convex hull cannot explain**.

The check below only confirms that the files were created. It does not write your answers for you.

In [ ]:
## Replace this with your own student ID.
STUDENT_ID = "demo"

REFLECTION = {
    "why_subsystems": "Write your own explanation here.",
    "one_limitation_of_0K_hull": "Write your own explanation here.",
    "selected_phase_ids": [],
    "lifeo2_entry_id": "The entry ID you picked",
    "lifeo2_e_hull_eV_atom": None,
}

In [ ]:
## Strip characters that cannot go in a filename. Fall back to demo if nothing is left.
safe_student_id = "".join(c for c in STUDENT_ID if c.isalnum() or c in "_-") or "demo"

attendance_csv = OUTPUT / f"attendance_{safe_student_id}_phases.csv"
attendance_png = OUTPUT / f"attendance_{safe_student_id}_phase_diagram.png"
attendance_json = OUTPUT / f"attendance_{safe_student_id}_reflection.json"

phase_df.to_csv(attendance_csv, index=False)
phase_fig.savefig(attendance_png, dpi=180, bbox_inches="tight")

## Record the calculation conditions and a timestamp so the result can be traced later.
attendance = {
    "student_id": STUDENT_ID,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "chemical_system": PD_ELEMENTS,
    "source": phase_source,
    "n_entries": len(pd_entries),
    "n_stable": len(phase_diagram.stable_entries),
    "energy_units": "eV/atom",
    "reflection": REFLECTION,
}
## ensure_ascii=False keeps non-ASCII text readable in the saved file.
attendance_json.write_text(json.dumps(attendance, ensure_ascii=False, indent=2), encoding="utf-8")

for file in [attendance_png, attendance_csv, attendance_json]:
    print(f"{'created' if file.exists() and file.stat().st_size > 0 else 'MISSING'}  {file}")

In [ ]:
## Two sanity checks on the calculation.
## Hull distances cannot be negative, and there must be one elemental reference per element.
print("No negative hull distances:", bool(phase_df["energy_above_hull_eV_atom"].ge(-1e-8).all()))
print("Elemental references:", len(phase_diagram.el_refs), "/", len(PD_ELEMENTS))
print("Student ID filled in:", STUDENT_ID != "demo")
print("Reflection written:", not REFLECTION["why_subsystems"].startswith("Write your own"))

### Homework
Narrow down candidates in a materials family you are interested in, and report both your reasoning
and the limits of the analysis.

- Adapt the AO example from A-6 to a different composition family or band gap window, and apply at
  least two conditions including stability.
- If API access is difficult, use `steel_strength` or the UCSB data and apply at least two conditions
  on properties and measurement conditions.
- Submit the executed notebook, the candidate CSV, and 5–8 sentences of reasoning.
  The reasoning must state the original row count, the row count after handling missing values, the
  number of rows passing your conditions, and the units.
- Draft rubric: reproducible code 40%, soundness of conditions and units 30%, interpretation and
  limitations 30%.

Once you are done, you can continue with the API examples in
`2026_2_Hands_on_session1_api_examples.ipynb` or with notebook 03 (Advanced).
Advanced is not graded.